# Shear slab — strain sweep

Analysis of **every** shear-strain level of a `shear_slab` sweep, overlaid so the whole stress–strain response
is visible at a glance (G, $N_1/N_2$, $D_c$ and κ vs strain). Set `STRAINS` and the run identifiers in **Config**,
run the **sync** cell once, then run everything. All analysis code lives in `scripts/lib/shear.py`; this
notebook only configures it and draws the ten figures. To look closely at one level, use
`shear_analysis_single.ipynb` instead. Method notes, file list and the physics behind each estimate are collected
in **Notes** at the very end.

**Two decks, one notebook.** Set `DECK` in **Config**: `"shear_slab"` (harmonic-bonded plates) or `"shear_slab_two_pist"` (the two-piston system: reservoirs at $P_{bath}$, a force on the polymer beads of the gel's two face layers steers each layer's centre of mass). Same strain-controlled protocol, `_g<strain>` files and figures. For the two-piston deck the dash-dot planes are the gel's Rg faces, the solid strain curve is the driven-layer (prescribed) strain, and the tracking-slab strain is one more strain measure; `G_plate` is the face-layer force per area over that strain. There is no unload phase, so the residual figure skips itself (as it does for `shear_slab` runs without `-var unload 1`).

**The strain G divides by (`G_STRAIN`, 2026-10-03).** As M in the compression notebooks (`M_STRAIN`), G and κ divide by the strain of the *network*: the interior slope of the steady cumulative displacement profile $u_x(z)$ (figure 1b), falling back to the Rg-based shear strain $(R_{g,xz}-R_{g,xz}^{\rm ref})/R_{g,zz}^2$, then the tracking slabs (two-piston deck), then the prescribed boundary strain — which is what runs of decks older than 2026-10-03 have. The strain's CI is folded into every G.


## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: shear.py (all shear analysis code) + triaxial.py (shared machinery)
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
import shear as sh
tri = importlib.reload(tri)
sh = importlib.reload(sh)              # pick up edits to lib/shear.py without a kernel restart
sh.setup_style()
print('analysis code: ', LIB / 'shear.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
cfg = sh.Config(
    DATANAME    = "final_config_slab_support_5beads_tall_rho04_nowalls_1.0_1.0_8000000_x_plates",
                                  # = DATANAME in shear_slab_two_pist_sweep.batch (isolated deck: the x_plates converter stem)
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    DECK        = "shear_slab_two_pist_iso",   # "shear_slab" (plate-driven) | "shear_slab_two_pist" (periodic two-piston slab, rigid-in-plane face layers;
                                        #   deck file shear_slab_two_pist_periodic.lmp) | "shear_slab_two_pist_iso" (ISOLATED gel,
                                        #   2026-10-13: x-face grips driven -/+z; lib relabels x <-> z, so "z" below = the gap)
    NSTEPS      = None,           # <steps> tag of the file names = NSTEPS of the batch (one tag for every level);
                                  # None resolves it from the files after the sync, an int pins it.
    RUN_ID      = "two_pist_rho04_iso_8M_3M_shear_5_sweep",   # local folder under flow_data_local/{shear,plots/shear}
                                  # (job 55082229, 2026-10-19: four cumulative levels, 3 M-step holds, REF_AVG_STEPS = 1 M)
    STRAINS     = ["0.01", "0.03", "0.05", "0.1"],   # = STRAINS_LIST=(...) in shear_slab_two_pist_sweep.batch, as strings
    # ---- measurement windows (the compression notebooks' values) ----
    plateau_frac      = 0.25,   # FIXED trailing fraction of the hold for the profile plateau means
    plateau_frac_auto = 0.45,   # LONGEST candidate trailing window for the series plateau (auto drift test)
    # ---- G as an increment from the gamma = 0 reference (as M and G in the compression notebooks) ----
    # ---- the strain G and kappa divide by (the compression notebooks' M_STRAIN) ----
    G_STRAIN = 'disp',          # 'disp' steady u_x(z) profile slope | 'rg' | 'trk' (two-piston deck) | 'applied' (prescribed);
                                # falls back down that chain when a file is missing (decks before 2026-10-03 -> applied)
    G_SUBTRACT_REF = True,      # needs the _ref files of shear_slab.lmp Phase 1.5 (runs since 2026-09-23);
                                # older runs fall back to absolute, flagged
    # ---- D_c from the UNLOAD (deck Phase 2c, UNLOAD=1 runs; files _u<γ>): the lag-profile fit ----
    DC_UNLOAD = True, DC_UNLOAD_N_MODES = 3, DC_TRIM_BINS = 1, DC_UNLOAD_SKIP = 0.0,
)
# Every other knob keeps its default (binWidth, plate_excl, wall_margin, n_curves, Ncount_min, dt_lj,
# ci_level, roll_win, P_BARO, DC_UNLOAD_BOUNDS, Expanse host/paths): see `sh.Config` in lib/shear.py, or pass them
# here as extra keyword arguments.

In [ ]:
# Pull the files this notebook reads (every level in STRAINS) from Expanse in ONE login (password + TOTP prompts).
# Logs in when ANY data file is missing locally (files a previous sync found absent on the
# cluster are remembered and do not re-prompt); FORCE_SYNC=True re-checks everything.
# SYNC=False never touches the network.
SYNC, FORCE_SYNC = True, False
if SYNC:
    sh.sync_from_expanse(cfg, levels=cfg.STRAINS + ['unload'], force=FORCE_SYNC)   # 'unload' = the Phase 4 files (optional)

In [ ]:
# Load + compute everything (a few seconds per level):
#   R       -- the gamma = 0 reference state (shared by every level)
#   LEVELS  -- one dict per strain level: strain, stresses, series plateau, G, N1/N2, P_th; with the deck's unloads (UNLOAD=1,
#              files _u<γ>) also L['U'] (each level's relaxation hold) and L['Dc'], kappa
R = sh.load_reference(cfg)
LEVELS = [L for L in (sh.load_level(cfg, R, lvl) for lvl in cfg.STRAINS) if L is not None]
assert LEVELS, 'no level loaded -- run the sync cell'
LEVELS.sort(key=lambda L: L['gamma'])
U = LEVELS[-1].get('U') if cfg.two_pist else sh.load_level(cfg, R, 'unload')   # the last level's unload (old deck: its single Phase 4 unload)
sh._xlim_from(R)
sh.print_summary(cfg, LEVELS)

## 2 · Figures

In [ ]:
# 1 · Strain diagnostic, all levels -- solid plate-based γ, dashed surface-COM γ; shaded = plateau windows
sh.fig_strain(cfg, R, LEVELS, stem='sweep_strain_diagnostic');

In [ ]:
# 1b · Steady displacement profiles u_x(z) of every level with their interior line fits: slope = the network strain G divides by
sh.fig_disp_profile(cfg, R, LEVELS, stem='sweep_disp_profile');


In [ ]:
# 2 · Total stress, final state of every level: σ^t_xz and the normal components / P_bath (compression-positive; solvent profile swap-corrected; reference dashed)
sh.fig_total_stress_sweep(cfg, R, LEVELS);


In [ ]:
# 2b · Normal stresses BEFORE shearing (γ = 0 reference window, totals, compression-positive): (a) bulk σ^t_xx, σ^t_yy, σ^t_zz
#      and P_th over the window with the lateral pre-stress ⟨σ^t_xx − σ^t_zz⟩, ⟨σ^t_yy − σ^t_zz⟩ (block bootstrap); (b) their window-mean
#      z-profiles.  The reservoirs must read P_bath in all three (fluid); a lateral component away from σ^t_zz inside the gel is a
#      pre-stress of the slab (periodic x,y / free z) -> transversely isotropic network, (C33 − C13)/2 of compression ≠ C44 of shear.
sh.fig_ref_normal_stress(cfg, R, stem='sweep_ref_normal_stress');


In [ ]:
# 3 · Network stress σ' = σ^t − p_pore (Terzaghi): σ'_xz(z) and σ'_xx(z), FINAL plateau state of every level + reference
sh.fig_network_stress_sweep(cfg, R, LEVELS);


In [ ]:
# 4 · Bulk total shear stress histories σ^t_xz(t), all levels (dotted = plateau value)
sh.fig_series_sweep(cfg, R, LEVELS);


In [ ]:
# 5 · Shear modulus G from σ'_xz vs held strain (increments from γ = 0): (a) with the absolute values, hollow, and the polymer-partial check;
#     (b) the estimates alone (network, series and, for runs of the 2026-09-25 deck, the plate x-force estimate)
sh.fig_G_sweep(cfg, R, LEVELS);


In [ ]:
# 5b · Stress vs strain: plateau σ'_xz per level + the γ = 0 reading (hollow); least-squares slopes = offset-free G, through-origin fit for comparison
_, G_fits = sh.fig_stress_strain_sweep(cfg, R, LEVELS)
print('stress-strain slopes (LS incl. gamma=0, through-origin):', {k: (sh.sig(v[0]), sh.sig(v[1])) for k, v in G_fits.items()})


In [ ]:
# 6 · Normal stress differences N1, N2 vs strain (polymer filled, total hollow; compression-positive)
sh.fig_normal_sweep(cfg, R, LEVELS);


In [ ]:
# 7 · Cooperative diffusivity from the UNLOADS (deck Phase 2c): D_c vs the strain released + the lag-profile fit of every level.
#     Skipped when the run has no unload files (shear_5_sweep was run with UNLOAD=0; shear_slab_two_pist_sweep.batch has UNLOAD=1).
sh.fig_Dc_sweep(cfg, R, LEVELS);
# 7b · FINE files (fast-drive runs with DISP_FINE_NFREQ > 0): mode relaxation times τ_k and their diffusive readings vs strain + a_k(t) per level;
#      skips itself for runs without the files
sh.fig_Dc_fine_sweep(cfg, R, LEVELS);


In [ ]:
# 8 · κ = D_c/G vs strain (unload D_c; network and series G)
sh.fig_kappa_sweep(cfg, R, LEVELS);

In [ ]:
# 9 · Thermodynamic pressure P_th = ⅓ tr(σ^t) over every hold (static-group bulk tensor; dotted = P_bath) -- see the single notebook's figure 3 for what 'bulk' means
sh.fig_thermo_pressure_sweep(cfg, R, LEVELS);


In [ ]:
# 10 · Residual stress after unloading the last level (its _u<γ> files; old deck: the Phase 4 _unload files): load → hold → unload → hold, and σ'_xz(z) reference / loaded / unloaded
sh.fig_residual(cfg, R, LEVELS[-1], U);



## Notes

Everything below is reference material — nothing above depends on reading it.

### How to use this notebook

1. **Config**: `DATANAME`, `INTERACTION`, `NSTEPS` name the run exactly as LAMMPS tagged its output files
   (`<DATANAME>_<INTERACTION>_<NSTEPS>`); `RUN_ID` is the local folder under `flow_data_local/{shear,plots/shear}/`.
   Every production file carries a `_g<level>` tag (even a single-level run); the `*_ref*` files of the γ = 0
   window are shared across levels. `NSTEPS = None` resolves the tag from the files (`cfg.tag_for()`).
2. **Sync**: one Expanse login pulls every file the notebook reads (`sh.sync_from_expanse`, the triaxial puller).
3. **Load**: `load_reference` builds the γ = 0 state `R`, `load_level` builds the level dict `L` (strain, stresses,
   series plateau, $G$, $N_1/N_2$, $P_{th}$, $D_c$, $\kappa$), `print_summary` prints the headline numbers and the
   hold-adequacy check.
4. **Figures**: one function call each; every figure is also saved as a PNG in `PLOT_DIR`.

`lib/shear.py` is shared with `shear_analysis_sweep.ipynb` so the definitions cannot drift between them, and it
imports its statistics, readers, palette and legend placement from `lib/triaxial.py`, so the shear numbers are
formed with the same code as the compression numbers.

### Files read

Into `flow_data_local/shear/<RUN_ID>/` (cluster `output_files/…`), all written by `shear_slab.lmp` (z = gap, x = shear since 2026-09-22):

| file pattern | content |
|---|---|
| `stress_tensor_{polymer,solvent}[_ref]_<sim>[_g<lvl>].dat` | bulk-integrated partial stress tensors (xx yy zz xy xz yz; `compute stress/atom NULL` → **kinetic term included**), one row per averaging block of the hold |
| `stress_profile_z_{polymer,solvent}[_ref]_…` | the same six components binned in z over the bulk region (reduced coordinate z/L_z, Ncount per bin) |
| `stress_series[_ref]_…` | fine block-averaged bulk stresses (xx yy zz xz, polymer \| solvent) every `series_freq` steps over drive + hold — the analogue of `piston_force_avg` |
| `shear_strain_…` | step, L_ref, plate separation, **plate-based γ** (drive + hold); `shear_strain_surface_…`: surface-COM γ (slip diagnostic) |
| `disp_x_polymer_cum_…_u<γ>` | polymer $u_x(z,t)$ of the level's unload (cumulative since γ = 0), for $D_c$ (2026-10-28; the hold-referenced `disp_x_polymer` file is gone) |
| `plate_pressure_…`, `box_dimensions_…`, `gel_dimensions_{bb,rg}_…`, `polymer_com_…` | plate planes, box, gel geometry (whole-run and per level) |

### Geometry, bulk region, windows

* Plates are normal to $z$ on the slab's faces, driven $\pm x$; the network is periodic in $x$ and $y$. Profiles
  are $z$-binned (`binWidth` = 2 σ) over the **bulk** region, `plate_excl` = 3 σ inside each plate plane; the
  **interior** used for every mean is trimmed to `wall_margin` = 4 σ inside the plates, the compression
  notebooks' `wall_margin`. Plate planes (dash-dot) are placed `plate_excl` outside the populated bins.
* Plateau window = the last `plateau_frac` = 25 % of the hold; evolution plots show the whole hold
  (`n_curves` snapshots, cividis, final curve bold black).
* The γ = 0 **reference** is Phase 1.5 of the deck: plates frozen, `ref_avg_steps` of NVT after the
  thermalisation, several snapshots so every reference number carries a CI — the analogue of the compression
  deck's ε = 0 window. The shear deck runs no barostat on the periodic-slab input (it arrives at P* = 1.5 from
  the aniso-NPH slab; the converter deletes the solvent the plates displace), so figure 10's bulk $P_{th}$
  is the check that the shear reference state is the compression runs' state.

### Strain

The strain is **prescribed**: `fix move` sets both plate positions, so γ = (x_top − x_bot − x₀)/plate_sep is exact;
the notebook uses its plateau mean (`L['gamma']`, the `held` value — the halt lands within ~0.0004 of the target
since `stress_freq` = 1000). The surface-COM strain of the polymer layer bonded to the plates (dashed in figure 1)
is a slip diagnostic: it lags γ only by the plate-bond stretch.

### Network stress and the shear modulus $G$

Definitions are those of `shear_analysis_single.ipynb` (Notes there, 2026-09-25): the **network stress** is the Terzaghi
split $\sigma'=\sigma^t-p_{\rm pore}$ (total minus the solvent-only value behind the plates, `P_PORE_MODE`; `'bath'`
fallback = $P_{\rm bath}$ on the diagonal, 0 off-diagonal while the deck profiles only the static bulk groups), so
$\sigma'_{xz}=\sigma^t_{xz}$ and $G=\Delta\langle\sigma'_{xz}\rangle/\gamma$ per level — **not** the polymer partial
$\sigma_{p,xz}$, which misses the solvent's half of the polymer–solvent contact virial (~9 % of $\sigma'_{xz}$; kept as the
hollow check). Normal components are compression-positive (triaxial convention); solvent profiles are swap-corrected for the
static `solv_bulk` group's exchange with the film behind the plates (`SWAP_CORR`). Two estimators per level, built like
$M_{\rm network}$ and $M_{\rm piston}$: **$G_{\rm network}$** (plateau $\sigma'_{xz}(z)$ over the interior bins, minus the
reference, over γ) and **$G_{\rm series}$** (bulk total $\sigma^t_{xz}(t)$ over the longest drift-free trailing window, block
bootstrap, minus the reference series, over γ).

With `G_SUBTRACT_REF=True` (default) every estimate is an **increment** from the γ = 0 reading — a slope, like the
compression $M$ and $G$ — and the absolute stress/γ values are drawn hollow in figure 6(a). $\sigma_{xz}(\gamma=0)$
vanishes by symmetry, so the subtraction mostly propagates the reference noise into the CI; runs older than
2026-09-23 have no `_ref` files and are reported absolute, flagged.

**Comparing with the compression G.** The compression notebooks form $G = (\sigma'_{zz}-\sigma'_{xx})/2\varepsilon$ from
the lateral network-stress anisotropy of the *same* 14000002 slab at the same P* = 1.5; the shear $G$ here is the direct
measurement. Both are increments from a reference window, both average the last 25 % of a hold over wall-trimmed
interior bins with t-interval CIs, and both quote a second, independent estimator (piston / series). Compare the
network estimates first; the strain ranges differ (compression ε = 0.05–0.2 vs shear γ = 0.1–0.3), so use the sweep
notebook's stress–strain slope if either shows nonlinearity.

### Normal stress differences (figure 7)

$N_1=\sigma_{xx}-\sigma_{yy}$, $N_2=\sigma_{yy}-\sigma_{zz}$ from the bulk tensors, plateau means with t-intervals over the
snapshots; both vanish for a linear isotropic network, so $N_i/\sigma_{xz}$ (printed by the summary) measures
the departure from linearity at this γ.

### Cooperative diffusivity $D_c$ and $\kappa$ — from the unload (2026-10-28)

* **Nothing is read from the hold's own transient any more.** The hold-onset `disp_x_polymer` file was a
  one-snapshot `displace/atom` reference that the block averages carried as a static pattern (the fits sat at
  the search bound), and the sheared gel shows no diffusive transient in the hold anyway (the fine-cadence
  files of October: the interior relaxed within the first block at the Kelvin–Voigt rate G/η_s).  Both files
  are gone from the deck.
* **The unload** (deck Phase 2c, `UNLOAD=1` in both batches): after every hold the grips are driven back at
  `vshear` until γ ≤ 0, frozen, and the gel relaxes for `UNLOAD_STEPS` (0 = `NSTEPS`); the level's file set
  is written again, tagged `_u<γ>`, and loaded as `L['U']` (`load_level(..., phase='u')`): residual stresses
  (figure 11) and the relaxation `D_c` is read from.
* **The fit** (`sh.fit_Dc_unload`): the cumulative displacement profile of the unload referenced to its own
  late plateau, `du(ẑ, t) = [u_late(ẑ) − u(ẑ, t)]/L`, the lag still to relax, over the blocks after the grips
  stopped, in `sin(kπẑ)` modes (u = 0 at both grips, back at γ = 0; `DC_UNLOAD_N_MODES`), each decaying as
  `exp(−k²π² D_c t/L²)` averaged over the block window, with a free constant per block (the floating gel's
  COM) and a free time-independent amplitude per mode (the late plateau has not fully relaxed).  One `D_c` for
  every mode (diffusive: τ_k = L²/(k²π² D_c), τ₁ = L²/(π² D_c)).  Beside it the **one-time alternative**, the
  same shapes all decaying at one τ (the Kelvin–Voigt rate of the two-fluid picture, no L² scaling); the ratio
  of the residuals decides which the data prefer (`pref`, printed and annotated).  A `D_c` at a `DC_UNLOAD_BOUNDS`
  edge is flagged, not reported.
* **κ = D_c/G** (figure 10) uses the unload `D_c` with the network and series `G` of the hold.
  `D_c^shear = G/ζ` against `D_c^comp = M/ζ` in compression with the same friction ζ, so this κ is the same
  hydraulic permeability as `D_c/M` there and `D_c^shear / D_c^comp = G/M`.
### Sweep conventions

* `STRAINS` must match `STRAINS_LIST=(...)` in `shear_slab.batch` (strings, so `"0.1"` finds `_g0.1` files). Levels
  whose core files are missing are skipped with a message; levels are sorted by their held γ.
* Colour = level (Wong palette), the γ = 0 reference profile is dashed black.
* Every per-level number ($G$, $N_i$, $D_c$, $\kappa$, $P_{th}$) is computed by the same `load_level` as the
  single-level notebook, so the two notebooks can never disagree on a level.
* Figure 5b: least-squares slope of the plateau $\sigma_{p,xz}$ vs held γ **including the γ = 0 reading** (dotted line;
  the compression notebooks' figure 7b) uses only differences between levels, so any constant offset an estimator
  carries cancels; the through-origin slope is printed next to it. Curvature of the points is the nonlinearity of
  the network — use the low-γ levels (or the slope) when comparing with the compression $G$ at ε ≤ 0.2.

### Colours and files

Wong (2011) palette; cividis for the time gradient; the bulk region is shaded grey. Figures are saved to
`flow_data_local/plots/shear/<RUN_ID>/` with the stems `strain_diagnostic` (tagged `_g<level>`),
`total_stress_evolution`, `partial_stress_evolution`, `network_stress_final`, `shear_stress_history`,
`G_comparison`, `normal_stress_differences`, `Dc_unload_fit`, `kappa`, `thermo_pressure`; this notebook's files are `sweep_*`.

*Restructured 2026-09-23 from the 13-step `shear_analysis.ipynb` (git history) to the layout of
`triaxial_compression_single.ipynb`; the G estimators were aligned with the compression notebooks' M/G machinery,
the strain became plate-based and the γ = 0 reference window was added to the deck the same day.*
